In [1]:
# --- CELL 0: MASTER KAGGLE SETUP ---

# 1. Install the required inference engine and vision utilities
!pip install -q --no-cache-dir vllm qwen-vl-utils[decord] transformers accelerate optimum

# 2. Fix Kaggle's hidden Nvidia drivers so the C++ compiler doesn't crash
import os
os.makedirs('/kaggle/working/compiler_links', exist_ok=True)
os.system("ln -sf /usr/local/nvidia/lib64/libcuda.so.1 /kaggle/working/compiler_links/libcuda.so")
print("[+] Environment rebuilt and C++ compiler linked successfully!")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.9/87.9 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 433.1/433.1 MB 277.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.3/194.3 kB 293.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 269.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 267.7/267.7 MB 112.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.8/7.8 MB 115.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.0/111.0 kB 357.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.4/45.4 kB 281.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.9/3.9 MB 116.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 132.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 114.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.2/161.2 kB 327.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
# cell 1: Universal Environment & Pipeline Setup
import os
import cv2
import json
import time
import torch
import numpy as np
from PIL import Image
from datasets import load_dataset
from decord import VideoReader, cpu


# 2. Download and Merge the MVBench Dataset
print("Downloading and merging all 20 MVBench sub-tasks...")

from datasets import concatenate_datasets

# The 20 specific task configs that make up the full 4000 queries
mvbench_configs = [
    'action_sequence', 'moving_count', 'action_prediction', 'episodic_reasoning', 
    'action_antonym', 'action_count', 'scene_transition', 'object_shuffle', 
    'object_existence', 'fine_grained_pose', 'unexpected_action', 'moving_direction', 
    'state_change', 'object_interaction', 'character_order', 'action_localization', 
    'counterfactual_inference', 'fine_grained_action', 'moving_attribute', 'egocentric_navigation'
]

# Download all 20 splits and staple them together
dataset_list = [load_dataset("OpenGVLab/MVBench", name=c, split="train") for c in mvbench_configs]
mvbench_dataset = concatenate_datasets(dataset_list)

print(f"Successfully loaded {len(mvbench_dataset)} total questions.")
VIDEO_DIR = '/kaggle/input/mvbench-raw-videos/'

# 3. Complete 5-Tier Dynamic Compression Pipeline (I/O & Memory Optimized)
def maximized_5_tier_compression(video_path, max_keyframes=8, l2_threshold=10.0, similarity_threshold=0.85):
    """
    Tier 1 & 2: Temporal Segmentation & DKS (proxy via structural clustering)
    Tier 3: Dynamic Supplementary Selection
    Tier 4: Adaptive Spatial Compression (Magnitude Pruning)
    Tier 5: Differential Feature Merging (DFM dynamically computed to avoid memory spikes)
    """
    vr = VideoReader(video_path, ctx=cpu(0))
    total_frames = len(vr)
    
    indices = np.linspace(0, total_frames - 1, num=max_keyframes * 4, dtype=int)
    raw_frames = vr.get_batch(indices).asnumpy()
    
    # FIX 1: Initialize list
    keyframes = []
    
    # Dynamic variables to prevent VRAM/RAM spikes from array accumulation
    merged_context_frame = None
    dropped_count = 0
    prev_hist = None
    
    for frame in raw_frames:
        gray = cv2.cvtColor(frame, cv2.COLOR_RGB2GRAY)
        
        # Tier 4 Proxy: Magnitude Pruning (Drop completely dark frames)
        if np.linalg.norm(gray) < l2_threshold:
            continue
            
        # FIX 2: Correct calcHist arguments
        hist = cv2.calcHist([gray], [0], None, [256], [0, 256])
        hist = cv2.normalize(hist, hist).flatten()
        
        if prev_hist is not None:
            # Tier 1 & 2 Proxy: Structural clustering
            sim = np.dot(hist, prev_hist) / (np.linalg.norm(hist) * np.linalg.norm(prev_hist))
            if sim < similarity_threshold:
                keyframes.append(frame)
                prev_hist = hist
            else:
                # Dynamically accumulate dropped frames to prevent memory spikes
                if merged_context_frame is None:
                    merged_context_frame = frame.astype(np.float32)
                else:
                    merged_context_frame += frame.astype(np.float32)
                dropped_count += 1
        else:
            keyframes.append(frame)
            prev_hist = hist
            
        if len(keyframes) == max_keyframes - 1:
            break
            
    # Tier 5 Proxy: Differential Feature Merging (DFM) 
    # Average the accumulated redundant frames into a single contextual representation
    if dropped_count > 0:
        merged_context_frame = (merged_context_frame / dropped_count).astype(np.uint8)
        keyframes.append(merged_context_frame)
            
    if len(keyframes) == 0:
        # FIX 3: Index the first frame properly
        keyframes = [raw_frames[0]]
        
    pil_frames = [Image.fromarray(frame) for frame in keyframes]
    return pil_frames

# 4. FLOP Standardization and OCR Routing Bypass
def extreme_token_compression(pil_frames, target_tokens=4):
    """Simulates extreme spatial pooling to force 14B model to adhere to SLM FLOP budget."""
    # FIX 4: Initialize list
    compressed_output = []
    for img in pil_frames:
        # Resize image drastically so the ViT encoder only generates 1 to 4 patches (e.g., 28x28)
        compressed_img = img.resize((28, 28), Image.Resampling.LANCZOS)
        compressed_output.append(compressed_img)
    return compressed_output

def is_ocr_task(question_text):
    """Keyword classifier to bypass FLOP standardization for text-heavy tasks."""
    ocr_keywords = ["read", "text", "sign", "written", "letters", "document", "words"]
    return any(word in question_text.lower() for word in ocr_keywords)

README.md: 0.00B [00:00, ?B/s]

action_sequence.json: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/200 [00:00<?, ? examples/s]

moving_count.json: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/200 [00:00<?, ? examples/s]

...

moving_attribute.json: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/200 [00:00<?, ? examples/s]

egocentric_navigation.json: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/200 [00:00<?, ? examples/s]

Successfully loaded 4000 total questions.


In [3]:
# cell 2: PERSON D EXECUTION SCRIPT
import os

# --- INJECT COMPILER PATH ---
# Force the C++ compiler to look in our custom folder FIRST
current_lib_path = os.environ.get("LIBRARY_PATH", "")
os.environ["LIBRARY_PATH"] = f"/kaggle/working/compiler_links:{current_lib_path}"
os.environ["VLLM_ENGINE_ITERATION_TIMEOUT_S"] = "600"
os.environ["VLLM_RPC_TIMEOUT"] = "600000"

from vllm import LLM, SamplingParams
from qwen_vl_utils import process_vision_info
from transformers import AutoProcessor
from huggingface_hub import hf_hub_download # <-- The missing import!
import time
import json
import torch
import zipfile
import shutil

START_IDX = 3000
END_IDX = 4000
BASE_DIR = "/kaggle/working"
TEMP_VIDEO_DIR = os.path.join(BASE_DIR, "jit_temp")
os.makedirs(TEMP_VIDEO_DIR, exist_ok=True)

# The 11 Master Zips on Hugging Face
master_zips = [
    "video/ssv2_video.zip", "video/tvqa.zip", "video/star.zip",
    "video/clevrer.zip", "video/data0613.zip", "video/FunQA_test.zip",
    "video/Moments_in_Time_Raw.zip", "video/perception.zip",
    "video/scene_qa.zip", "video/sta.zip", "video/vlnqa.zip"
]

# --- Model Initialization ---
llm = LLM(
    model="Qwen/Qwen2.5-VL-7B-Instruct-AWQ",
    tensor_parallel_size=2,
    max_model_len=4096,     
    gpu_memory_utilization=0.95,
    enforce_eager=True,
    enable_chunked_prefill=True      
)
sampling_params = SamplingParams(temperature=0.0, max_tokens=128)
processor = AutoProcessor.from_pretrained("Qwen/Qwen2.5-VL-7B-Instruct-AWQ")
results_log = []

# Resume functionality: Don't repeat questions we already answered!
if os.path.exists("/kaggle/working/person_D_results.json"):
    with open("/kaggle/working/person_D_results.json", "r") as f:
        results_log = json.load(f)
processed_ids = {item["id"] for item in results_log}

print(f"\nStarting Master Zip JIT Evaluation ({START_IDX} to {END_IDX - 1})...")

# --- THE REVERSE JIT LOOP ---
for zip_name in master_zips:
    print(f"\n[========== Fetching Master Zip: {zip_name} ==========]")
    try:
        zip_path = hf_hub_download(
            repo_id="OpenGVLab/MVBench",
            filename=zip_name,
            repo_type="dataset",
            local_dir=BASE_DIR
        )
    except Exception as e:
        print(f"Failed to download {zip_name}: {e}")
        continue
        
    with zipfile.ZipFile(zip_path, 'r') as zf:
        # Map bare filenames to their exact internal path in the zip
        zip_contents = {os.path.basename(f): f for f in zf.namelist() if f.endswith(('.mp4', '.mkv', '.webm', '.avi'))}
        print(f"Found {len(zip_contents)} videos in archive. Scanning for Person D's assigned questions...")
        
        matches_found = 0
        for i in range(START_IDX, END_IDX):
            if i in processed_ids:
                continue
                
            sample = mvbench_dataset[i]
            question = sample['question']
            candidates = sample['candidates']
            video_name = os.path.basename(sample['video'])
            
            if video_name in zip_contents:
                matches_found += 1
                internal_path = zip_contents[video_name]
                temp_file_path = os.path.join(TEMP_VIDEO_DIR, "current_eval.mp4")
                
                # 1. Extract ONE video temporarily
                with zf.open(internal_path) as source, open(temp_file_path, "wb") as target:
                    shutil.copyfileobj(source, target)
                
                # 2. RUN MODEL INFERENCE ON TEMP FILE
                options_text = "\n".join([f"({chr(65+j)}) {opt}" for j, opt in enumerate(candidates)])
                full_prompt = f"{question}\n{options_text}\nPlease output the correct option strictly as a single letter."
                
                try:
                    compressed_frames = maximized_5_tier_compression(temp_file_path)
                    if not is_ocr_task(question):
                        final_frames = extreme_token_compression(compressed_frames, target_tokens=4)
                    else:
                        final_frames = compressed_frames
                        
                    messages = [{"role": "user", "content": [
                        {"type": "video", "video": final_frames},
                        {"type": "text", "text": full_prompt}
                    ]}]
                    
                    prompt = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
                    image_inputs, video_inputs = process_vision_info(messages)
                    
                    outputs = llm.generate({"prompt": prompt, "multi_modal_data": {"video": video_inputs}}, sampling_params=sampling_params, use_tqdm=False)
                    response = outputs[0].outputs[0].text.strip()
                    
                except Exception as e:
                    print(f"Failed on query {i}: {e}")
                    response = "ERROR"
                
                # Log Result
                gt_index = candidates.index(sample['answer'])
                gt_letter = chr(65 + gt_index)
                
                results_log.append({
                    "id": i, "question": question, "response": response, 
                    "ground_truth_letter": gt_letter, "ground_truth_text": sample['answer'],
                    "source_zip": zip_name
                })
                processed_ids.add(i)
                
                # 3. DELETE THE TEMP VIDEO IMMEDIATELY
                if os.path.exists(temp_file_path):
                    os.remove(temp_file_path)
                
                # Save checkpoint frequently
                if len(results_log) % 10 == 0:
                    with open("/kaggle/working/person_D_results.json", "w") as f:
                        json.dump(results_log, f)
                        
        print(f"Processed {matches_found} assigned questions from {zip_name}.")
                        
    # 4. DELETE THE GIANT ZIP AFTER ALL MATCHING QUESTIONS ARE DONE
    print(f"[-] Deleting {zip_name} to free disk space.")
    if os.path.exists(zip_path):
        os.remove(zip_path)
    shutil.rmtree(os.path.join(BASE_DIR, "video"), ignore_errors=True)

# Final save
with open("/kaggle/working/person_D_results.json", "w") as f:
    json.dump(results_log, f)
print("\nPerson D execution complete!")

INFO 04-18 10:47:21 [utils.py:233] non-default args: {'max_model_len': 4096, 'tensor_parallel_size': 2, 'gpu_memory_utilization': 0.95, 'disable_log_stats': True, 'enforce_eager': True, 'enable_chunked_prefill': True, 'model': 'Qwen/Qwen2.5-VL-7B-Instruct-AWQ'}


config.json: 0.00B [00:00, ?B/s]

preprocessor_config.json:   0%|          | 0.00/575 [00:00<?, ?B/s]

INFO 04-18 10:47:41 [model.py:549] Resolved architecture: Qwen2_5_VLForConditionalGeneration
WARNING 04-18 10:47:41 [model.py:1963] Your device 'Tesla T4' (with compute capability 7.5) doesn't support torch.bfloat16. Falling back to torch.float16 for compatibility.
WARNING 04-18 10:47:41 [model.py:2016] Casting torch.bfloat16 to torch.float16.
INFO 04-18 10:47:41 [model.py:1678] Using max model len 4096
INFO 04-18 10:47:41 [awq_marlin.py:245] The model is convertible to awq_marlin during runtime. Using awq_marlin kernel.
INFO 04-18 10:47:42 [scheduler.py:238] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 04-18 10:47:42 [vllm.py:790] Asynchronous scheduling is enabled.
WARNING 04-18 10:47:42 [vllm.py:848] Enforce eager set, disabling torch.compile and CUDAGraphs. This is equivalent to setting -cc.mode=none -cc.cudagraph_mode=none
WARNING 04-18 10:47:42 [vllm.py:859] Inductor compilation was disabled by user settings, optimizations settings that are only active during

tokenizer_config.json: 0.00B [00:00, ?B/s]

...

Found 200 videos in archive. Scanning for Person D's assigned questions...
Processed 0 assigned questions from video/scene_qa.zip.
[-] Deleting video/scene_qa.zip to free disk space.

[========== Fetching Master Zip: video/sta.zip ==========]


video/sta.zip:   0%|          | 0.00/1.09G [00:00<?, ?B/s]

Found 200 videos in archive. Scanning for Person D's assigned questions...
Processed 188 assigned questions from video/sta.zip.
[-] Deleting video/sta.zip to free disk space.

[========== Fetching Master Zip: video/vlnqa.zip ==========]


video/vlnqa.zip:   0%|          | 0.00/40.3M [00:00<?, ?B/s]

Found 200 videos in archive. Scanning for Person D's assigned questions...
Processed 200 assigned questions from video/vlnqa.zip.
[-] Deleting video/vlnqa.zip to free disk space.

Person D execution complete!


In [4]:
import json
import re
import pandas as pd
from collections import defaultdict
from datasets import load_dataset, concatenate_datasets

# 1. Configuration
RESULTS_FILE = "/kaggle/working/person_D_results.json"

# The 20 sub-tasks of MVBench
mvbench_configs = [
    'action_sequence', 'moving_count', 'action_prediction', 'episodic_reasoning', 
    'action_antonym', 'action_count', 'scene_transition', 'object_shuffle', 
    'object_existence', 'fine_grained_pose', 'unexpected_action', 'moving_direction', 
    'state_change', 'object_interaction', 'character_order', 'action_localization', 
    'counterfactual_inference', 'fine_grained_action', 'moving_attribute', 'egocentric_navigation'
]

print("Loading dataset metadata for task mapping...")
# We only load the text metadata (fast, no videos) to map IDs to their task categories
dataset_list = [load_dataset("OpenGVLab/MVBench", name=c, split="train") for c in mvbench_configs]
full_metadata = concatenate_datasets(dataset_list)

print("Loading model predictions...")
with open(RESULTS_FILE, "r") as f:
    results = json.load(f)

# 2. Strict Regex Parser (Research Standard)
def extract_choice(response_text):
    """
    LLMs are chatty. Even with strict prompts, they might output:
    'A', '(A)', 'The answer is B', or 'Option C is correct.'
    This regex extracts the first standalone A, B, C, D, or E.
    """
    if not response_text:
        return "INVALID"
    
    # Look for (A), (B), etc.
    match = re.search(r'\(([A-E])\)', response_text, re.IGNORECASE)
    if match:
        return match.group(1).upper()
        
    # Look for standalone A, B, C, D, E at the start or ends of strings
    match = re.search(r'\b([A-E])\b', response_text, re.IGNORECASE)
    if match:
        return match.group(1).upper()
        
    return "INVALID"

# 3. Evaluation Engine (Updated for 20-Task grouping)
task_metrics = defaultdict(lambda: {"correct": 0, "total": 0})
total_correct = 0

for item in results:
    q_id = item["id"]
    raw_response = item["response"]
    ground_truth = item["ground_truth_letter"]
    
    # Each of the 20 tasks has exactly 200 questions. 
    # We can mathematically determine the task name based on the ID!
    task_index = q_id // 200 
    task_name = mvbench_configs[task_index]
    
    predicted_letter = extract_choice(raw_response)
    if predicted_letter == ground_truth:
        task_metrics[task_name]["correct"] += 1
        total_correct += 1
        
    task_metrics[task_name]["total"] += 1

# 4. Generate Research Table
print("\n" + "="*50)
print(f"{'MVBench Sub-Task':<30} | {'Accuracy'}")
print("="*50)

# Sort alphabetically for clean reporting
report_data = []
for task, metrics in sorted(task_metrics.items()):
    acc = (metrics["correct"] / metrics["total"]) * 100
    report_data.append({"Task": task, "Accuracy (%)": round(acc, 2), "N": metrics["total"]})
    print(f"{task:<30} | {acc:>5.2f}%  ({metrics['correct']}/{metrics['total']})")

overall_acc = (total_correct / total_evaluated) * 100
print("="*50)
print(f"{'OVERALL ACCURACY':<30} | {overall_acc:>5.2f}%  ({total_correct}/{total_evaluated})")
print("="*50)

# Save the final metrics to disk for your research paper/report
df = pd.DataFrame(report_data)
df.to_csv("/kaggle/working/person_D_metrics.csv", index=False)
print("\nDetailed metrics saved to /kaggle/working/person_D_metrics.csv")

Loading dataset metadata for task mapping...
Loading model predictions...

MVBench Sub-Task               | Accuracy
action_localization            | 30.00%  (60/200)
counterfactual_inference       | 38.50%  (77/200)
egocentric_navigation          | 36.00%  (72/200)
fine_grained_action            | 38.00%  (76/200)
moving_attribute               | 51.00%  (102/200)


NameError: name 'total_evaluated' is not defined